<a href="https://colab.research.google.com/github/MrSuperfluous/SummerAnal/blob/main/Summer_Anal_AI_Planet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from xgboost import XGBClassifier

# 1. Load Data
def load_data(train_path, test_path):
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    return train, test

# 2. Data Cleaning / Outlier Removal (without dropping NaNs)
def clean_data(df):
    num_feats = ['RIDAGEYR','PAQ605','BMXBMI','LBXGLU','LBXGLT','LBXIN']
    iso = IsolationForest(contamination=0.02, random_state=42)
    df_iso = df[num_feats].copy()
    for col in num_feats:
        df_iso[col] = df_iso[col].fillna(df_iso[col].median())
    mask = iso.fit_predict(df_iso)
    df = df[mask == 1]

    for col in num_feats:
        if col in df.columns:
            low, high = df[col].quantile([0.01, 0.99])
            df[col] = df[col].clip(low, high)
    return df

# 3. Preprocessing + Denoising Pipeline
def make_pipeline():
    num_feats = ['RIDAGEYR','PAQ605','BMXBMI','LBXGLU','LBXGLT','LBXIN']
    cat_feats = ['RIAGENDR','DIQ010']

    num_pipe = Pipeline([
        ('imputer', KNNImputer(n_neighbors=5)),
        ('scaler', StandardScaler())
    ])
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ])

    from sklearn.compose import ColumnTransformer
    preprocessor = ColumnTransformer([
        ('num', num_pipe, num_feats),
        ('cat', cat_pipe, cat_feats)
    ])

    pca = PCA(n_components=0.95, random_state=42)

    def preprocess_and_denoise(X, fit=True):
        if fit:
            X_transformed = preprocessor.fit_transform(X)
            X_denoised = pca.fit_transform(X_transformed)
        else:
            X_transformed = preprocessor.transform(X)
            X_denoised = pca.transform(X_transformed)
        return X_denoised

    return preprocessor, pca, preprocess_and_denoise

# 4. Model Training Function
def train_and_evaluate(X, y, model, params=None):
    if params:
        grid = GridSearchCV(model, params, cv=5, scoring='roc_auc', n_jobs=-1, error_score='raise')
        grid.fit(X, y)
        best = grid.best_estimator_
    else:
        best = model
        best.fit(X, y)
    y_pred = best.predict(X)
    print(classification_report(y, y_pred))
    print("ROC AUC:", roc_auc_score(y, best.predict_proba(X)[:,1]))
    return best

# 5. Main Execution
if __name__ == '__main__':
    train_df, test_df = load_data('train.csv', 'test.csv')

    train_df = clean_data(train_df)
    test_df = clean_data(test_df)

    # Drop missing target labels only
    train_df = train_df.dropna(subset=['age_group'])

    # Map age_group to 0/1
    train_df['age_group'] = train_df['age_group'].map({'Adult': 0, 'Senior': 1})

    X_train_full = train_df.drop(['SEQN', 'age_group'], axis=1)
    y_train_full = train_df['age_group'].astype(int)
    X_test_feat = test_df.drop(['SEQN'], axis=1)

    preprocessor, pca, preprocess_fn = make_pipeline()

    X_train_trans = preprocess_fn(X_train_full, fit=True)
    X_test_trans = preprocess_fn(X_test_feat, fit=False)

    # Final NaN check and replace (fail-safe)
    X_train_trans = np.nan_to_num(X_train_trans)
    X_test_trans = np.nan_to_num(X_test_trans)

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_trans, y_train_full, test_size=0.2, stratify=y_train_full, random_state=42
    )

    xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
    xgb_params = {'n_estimators':[100,200],'max_depth':[3,5],'learning_rate':[0.01,0.1]}
    print("Training XGBoost...")
    best_model = train_and_evaluate(X_tr, y_tr, xgb, xgb_params)

    print("Validation Metrics:")
    y_val_pred = best_model.predict(X_val)
    print(classification_report(y_val, y_val_pred))
    print("ROC AUC:", roc_auc_score(y_val, best_model.predict_proba(X_val)[:,1]))

    submission = pd.DataFrame({
        'age_group': best_model.predict(X_test_trans)
    })
    submission.to_csv('hky1.csv', index=False)
    print("submission.csv created")

/tmp/ipython-input-17-917958496.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].clip(low, high)
/tmp/ipython-input-17-917958496.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].clip(low, high)
/tmp/ipython-input-17-917958496.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/use

Training XGBoost...
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1284
           1       1.00      1.00      1.00       246

    accuracy                           1.00      1530
   macro avg       1.00      1.00      1.00      1530
weighted avg       1.00      1.00      1.00      1530

ROC AUC: 1.0
Validation Metrics:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98       322
           1       0.92      0.89      0.90        61

    accuracy                           0.97       383
   macro avg       0.95      0.93      0.94       383
weighted avg       0.97      0.97      0.97       383

ROC AUC: 0.9918541900010182
submission.csv created


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:40:18] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import zipfile # Import the zipfile module

# Load the training data
train_df = pd.read_csv('train.csv')

# Define features
numeric_feats = ['RIDAGEYR', 'PAQ605', 'BMXBMI', 'LBXGLU', 'LBXGLT', 'LBXIN']
cat_feats = ['RIAGENDR', 'DIQ010']

# Create a directory to save plots if it doesn't exist
plots_dir = 'plots'
if not os.path.exists(plots_dir):
    os.makedirs(plots_dir)

# 1. Histograms for numeric features
for i, feat in enumerate(numeric_feats):
    plt.figure()
    train_df[feat].hist()
    plt.title(f'Distribution of {feat}')
    plt.xlabel(feat)
    plt.ylabel('Frequency')
    # Save the plot instead of showing
    plt.savefig(os.path.join(plots_dir, f'histogram_{i+1}_{feat}.png'))
    plt.close() # Close the plot to free memory

# 2. Boxplots for numeric features
for i, feat in enumerate(numeric_feats):
    plt.figure()
    plt.boxplot(train_df[feat].dropna(), vert=True)
    plt.title(f'Boxplot of {feat}')
    plt.ylabel(feat)
    # Save the plot instead of showing
    plt.savefig(os.path.join(plots_dir, f'boxplot_{i+1}_{feat}.png'))
    plt.close() # Close the plot to free memory

# 3. Bar charts for categorical features
for i, feat in enumerate(cat_feats):
    plt.figure()
    train_df[feat].value_counts().sort_index().plot(kind='bar')
    plt.title(f'Counts of {feat}')
    plt.xlabel(feat)
    plt.ylabel('Count')
    # Save the plot instead of showing
    plt.savefig(os.path.join(plots_dir, f'bar_chart_{i+1}_{feat}.png'))
    plt.close() # Close the plot to free memory

# 4. Missing value counts
plt.figure()
missing_counts = train_df.isnull().sum()
missing_counts.plot(kind='bar')
plt.title('Missing Values per Feature')
plt.xlabel('Feature')
plt.ylabel('Count of NaNs')
# Save the plot instead of showing
plt.savefig(os.path.join(plots_dir, 'missing_values_bar_chart.png'))
plt.close() # Close the plot to free memory

# 5. Correlation matrix heatmap for numeric features
plt.figure()
corr = train_df[numeric_feats].corr()

# Create figure and axes explicitly before calling matshow
fig, ax = plt.subplots()
# Remove fignum argument when calling matshow on axes object
cax = ax.matshow(corr)

plt.title('Correlation Matrix of Numeric Features', pad=20)
# Set tick locations and labels on the axes object
ax.set_xticks(range(len(numeric_feats)))
ax.set_xticklabels(numeric_feats, rotation=90)
ax.set_yticks(range(len(numeric_feats)))
ax.set_yticklabels(numeric_feats)

fig.colorbar(cax) # Add colorbar to the figure using the matshow output

# Save the plot instead of showing
plt.savefig(os.path.join(plots_dir, 'correlation_heatmap.png'))
plt.close() # Close the plot to free memory

print(f"All plots saved to the '{plots_dir}' directory.")

# Create a zip archive of the plots directory
zip_filename = 'plots_archive.zip'
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(plots_dir):
        for file in files:
            file_path = os.path.join(root, file)
            # Add file to zip, maintaining the directory structure within the zip
            zipf.write(file_path, os.path.relpath(file_path, os.path.dirname(plots_dir)))

print(f"Created zip archive: {zip_filename}")

All plots saved to the 'plots' directory.
Created zip archive: plots_archive.zip


<Figure size 640x480 with 0 Axes>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.ensemble import IsolationForest
from xgboost import XGBClassifier

# 1. Load Data
def load_data(train_path, test_path):
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    return train, test

# 2. Data Cleaning / Outlier Removal (without dropping NaNs)
def clean_data(df):
    num_feats = ['RIDAGEYR','PAQ605','BMXBMI','LBXGLU','LBXGLT','LBXIN']
    df_copy = df.copy()
    df_iso = df_copy[num_feats].copy()
    for col in num_feats:
        df_iso[col] = df_iso[col].fillna(df_iso[col].median())
    mask = IsolationForest(contamination=0.02, random_state=42).fit_predict(df_iso)
    # No filtering: preserve all rows

    for col in num_feats:
        if col in df_copy.columns:
            low, high = df_copy[col].quantile([0.01, 0.99])
            df_copy[col] = df_copy[col].clip(lower=low, upper=high)

    return df_copy

# 3. Preprocessing Pipeline (No PCA)
def make_pipeline():
    num_feats = ['RIDAGEYR','PAQ605','BMXBMI','LBXGLU','LBXGLT','LBXIN','BMI_GLU','INSULIN_RATIO']
    cat_feats = ['RIAGENDR','DIQ010']

    num_pipe = Pipeline([
        ('imputer', KNNImputer(n_neighbors=5)),
        ('scaler', StandardScaler())
    ])
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ])

    from sklearn.compose import ColumnTransformer
    preprocessor = ColumnTransformer([
        ('num', num_pipe, num_feats),
        ('cat', cat_pipe, cat_feats)
    ])

    def preprocess(X, fit=True):
        return preprocessor.fit_transform(X) if fit else preprocessor.transform(X)

    return preprocessor, preprocess

# 4. Model Training Function
def train_and_evaluate(X, y, model, params=None):
    if params:
        grid = GridSearchCV(model, params, cv=5, scoring='roc_auc', n_jobs=-1, error_score='raise')
        grid.fit(X, y)
        best = grid.best_estimator_
    else:
        best = model
        best.fit(X, y)
    y_pred = best.predict(X)
    print(classification_report(y, y_pred))
    print("ROC AUC:", roc_auc_score(y, best.predict_proba(X)[:,1]))
    return best

# 5. Ensemble Prediction via Cross Validation
def cross_val_predict_average(model, X, y, X_test, folds=5):
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)
    test_preds = np.zeros((X_test.shape[0],))
    for train_idx, _ in skf.split(X, y):
        X_tr, y_tr = X[train_idx], y[train_idx]
        model_ = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
        model_.fit(X_tr, y_tr)
        test_preds += model_.predict_proba(X_test)[:,1] / folds
    return test_preds

# 6. Main Execution
if __name__ == '__main__':
    train_df, test_df = load_data('train.csv', 'test.csv')

    # Feature engineering
    for df in [train_df, test_df]:
        df['BMI_GLU'] = df['BMXBMI'] * df['LBXGLU']
        df['INSULIN_RATIO'] = df['LBXIN'] / (df['LBXGLU'] + 1)

    train_df = clean_data(train_df)
    test_df = clean_data(test_df)  # Avoid filtering rows in test set

    train_df = train_df.dropna(subset=['age_group'])
    train_df['age_group'] = train_df['age_group'].map({'Adult': 0, 'Senior': 1})

    X_train_full = train_df.drop(['SEQN', 'age_group'], axis=1)
    y_train_full = train_df['age_group'].astype(int)
    X_test_feat = test_df.drop(['SEQN'], axis=1)

    preprocessor, preprocess_fn = make_pipeline()
    X_train_trans = preprocess_fn(X_train_full, fit=True)
    X_test_trans = preprocess_fn(X_test_feat, fit=False)

    X_train_trans = np.nan_to_num(X_train_trans)
    X_test_trans = np.nan_to_num(X_test_trans)

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_trans, y_train_full, test_size=0.2, stratify=y_train_full, random_state=42
    )

    xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
    xgb_params = {
        'n_estimators': [100, 200, 300],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.05, 0.1],
        'subsample': [0.7, 0.9, 1.0],
        'colsample_bytree': [0.7, 0.9, 1.0]
    }

    print("Training XGBoost...")
    best_model = train_and_evaluate(X_tr, y_tr, xgb, xgb_params)

    print("Validation Metrics:")
    y_val_pred = best_model.predict(X_val)
    print(classification_report(y_val, y_val_pred))
    print("ROC AUC:", roc_auc_score(y_val, best_model.predict_proba(X_val)[:,1]))

    print("Generating ensemble prediction on test set...")
    test_probs = cross_val_predict_average(best_model, X_train_trans, y_train_full.to_numpy(), X_test_trans, folds=5)
    test_preds = (test_probs > 0.5).astype(int)

    submission = pd.DataFrame({
        'age_group': test_preds
    })
    submission.to_csv('hcky3.csv', index=False)
    print("submission.csv created")


Training XGBoost...


/tmp/ipython-input-19-2998029889.py:97: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['age_group'] = train_df['age_group'].map({'Adult': 0, 'Senior': 1})
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:52:06] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:52:06] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1310
           1       1.00      1.00      1.00       251

    accuracy                           1.00      1561
   macro avg       1.00      1.00      1.00      1561
weighted avg       1.00      1.00      1.00      1561

ROC AUC: 1.0
Validation Metrics:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       328
           1       1.00      1.00      1.00        63

    accuracy                           1.00       391
   macro avg       1.00      1.00      1.00       391
weighted avg       1.00      1.00      1.00       391

ROC AUC: 1.0
Generating ensemble prediction on test set...
submission.csv created


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.ensemble import IsolationForest
from xgboost import XGBClassifier

# 1. Load Data
def load_data(train_path, test_path):
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    return train, test

# 2. Data Cleaning / Outlier Removal (without dropping NaNs)
def clean_data(df):
    num_feats = ['PAQ605','BMXBMI','LBXGLU','LBXGLT','LBXIN']
    df_copy = df.copy()
    df_iso = df_copy[num_feats].copy()
    for col in num_feats:
        df_iso[col] = df_iso[col].fillna(df_iso[col].median())
    mask = IsolationForest(contamination=0.02, random_state=42).fit_predict(df_iso)
    # No filtering: preserve all rows

    for col in num_feats:
        if col in df_copy.columns:
            low, high = df_copy[col].quantile([0.01, 0.99])
            df_copy[col] = df_copy[col].clip(lower=low, upper=high)

    return df_copy

# 3. Preprocessing Pipeline (No PCA)
def make_pipeline():
    num_feats = ['PAQ605','BMXBMI','LBXGLU','LBXGLT','LBXIN','BMI_GLU','INSULIN_RATIO']
    cat_feats = ['RIAGENDR','DIQ010']

    num_pipe = Pipeline([
        ('imputer', KNNImputer(n_neighbors=5)),
        ('scaler', StandardScaler())
    ])
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ])

    from sklearn.compose import ColumnTransformer
    preprocessor = ColumnTransformer([
        ('num', num_pipe, num_feats),
        ('cat', cat_pipe, cat_feats)
    ])

    def preprocess(X, fit=True):
        return preprocessor.fit_transform(X) if fit else preprocessor.transform(X)

    return preprocessor, preprocess

# 4. Model Training Function
def train_and_evaluate(X, y, model, params=None):
    if params:
        grid = GridSearchCV(model, params, cv=5, scoring='roc_auc', n_jobs=-1, error_score='raise')
        grid.fit(X, y)
        best = grid.best_estimator_
    else:
        best = model
        best.fit(X, y)
    y_pred = best.predict(X)
    print(classification_report(y, y_pred))
    print("ROC AUC:", roc_auc_score(y, best.predict_proba(X)[:,1]))
    return best

# 5. Ensemble Prediction via Cross Validation
def cross_val_predict_average(model, X, y, X_test, folds=5):
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)
    test_preds = np.zeros((X_test.shape[0],))
    for train_idx, _ in skf.split(X, y):
        X_tr, y_tr = X[train_idx], y[train_idx]
        model_ = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
        model_.fit(X_tr, y_tr)
        test_preds += model_.predict_proba(X_test)[:,1] / folds
    return test_preds

# 6. Main Execution
if __name__ == '__main__':
    train_df, test_df = load_data('train.csv', 'test.csv')

    # Feature engineering
    for df in [train_df, test_df]:
        df['BMI_GLU'] = df['BMXBMI'] * df['LBXGLU']
        df['INSULIN_RATIO'] = df['LBXIN'] / (df['LBXGLU'] + 1)

    train_df = clean_data(train_df)
    test_df = clean_data(test_df)  # Avoid filtering rows in test set

    train_df = train_df.dropna(subset=['age_group'])
    train_df['age_group'] = train_df['age_group'].map({'Adult': 0, 'Senior': 1})

    X_train_full = train_df.drop(['SEQN', 'age_group'], axis=1)
    y_train_full = train_df['age_group'].astype(int)
    X_test_feat = test_df.drop(['SEQN'], axis=1)

    preprocessor, preprocess_fn = make_pipeline()
    X_train_trans = preprocess_fn(X_train_full, fit=True)
    X_test_trans = preprocess_fn(X_test_feat, fit=False)

    X_train_trans = np.nan_to_num(X_train_trans)
    X_test_trans = np.nan_to_num(X_test_trans)

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_trans, y_train_full, test_size=0.2, stratify=y_train_full, random_state=42
    )

    xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
    xgb_params = {
        'n_estimators': [100, 200, 300, 500],
        'max_depth': [3, 5, 7, 9],
        'learning_rate': [0.01, 0.05, 0.1],
        'subsample': [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0]
    }

    print("Training XGBoost...")
    best_model = train_and_evaluate(X_tr, y_tr, xgb, xgb_params)

    print("Validation Metrics:")
    y_val_pred = best_model.predict(X_val)
    print(classification_report(y_val, y_val_pred))
    print("ROC AUC:", roc_auc_score(y_val, best_model.predict_proba(X_val)[:,1]))

    print("Generating ensemble prediction on test set...")
    test_probs = cross_val_predict_average(best_model, X_train_trans, y_train_full.to_numpy(), X_test_trans, folds=5)
    test_preds = (test_probs > 0.5).astype(int)

    submission = pd.DataFrame({
        'age_group': test_preds
    })
    submission.to_csv('submission.csv', index=False)
    print("submission.csv created")


Training XGBoost...


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [07:23:19] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [07:23:19] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


              precision    recall  f1-score   support

           0       0.85      0.99      0.92      1310
           1       0.76      0.09      0.16       251

    accuracy                           0.85      1561
   macro avg       0.80      0.54      0.54      1561
weighted avg       0.84      0.85      0.79      1561

ROC AUC: 0.8472491712539156
Validation Metrics:
              precision    recall  f1-score   support

           0       0.84      0.99      0.91       328
           1       0.50      0.05      0.09        63

    accuracy                           0.84       391
   macro avg       0.67      0.52      0.50       391
weighted avg       0.79      0.84      0.78       391

ROC AUC: 0.6869434765776229
Generating ensemble prediction on test set...


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [07:23:21] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


submission.csv created


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.ensemble import IsolationForest
from xgboost import XGBClassifier

# 1. Load Data
def load_data(train_path, test_path):
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    return train, test

# 2. Data Cleaning / Outlier Removal (without dropping NaNs)
def clean_data(df):
    num_feats = ['PAQ605','BMXBMI','LBXGLU','LBXGLT','LBXIN']
    df_copy = df.copy()
    df_iso = df_copy[num_feats].copy()
    for col in num_feats:
        df_iso[col] = df_iso[col].fillna(df_iso[col].median())
    mask = IsolationForest(contamination=0.02, random_state=42).fit_predict(df_iso)
    # No filtering: preserve all rows

    for col in num_feats:
        if col in df_copy.columns:
            low, high = df_copy[col].quantile([0.01, 0.99])
            df_copy[col] = df_copy[col].clip(lower=low, upper=high)

    return df_copy

# 3. Preprocessing Pipeline (No PCA)
def make_pipeline():
    num_feats = ['PAQ605','BMXBMI','LBXGLU','LBXGLT','LBXIN','BMI_GLU','INSULIN_RATIO','GLU_TOL_RATIO','INSULIN_GLU_RATIO','ACTIVITY_DIABETES']
    cat_feats = ['RIAGENDR','DIQ010']

    num_pipe = Pipeline([
        ('imputer', KNNImputer(n_neighbors=5)),
        ('scaler', StandardScaler())
    ])
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ])

    from sklearn.compose import ColumnTransformer
    preprocessor = ColumnTransformer([
        ('num', num_pipe, num_feats),
        ('cat', cat_pipe, cat_feats)
    ])

    def preprocess(X, fit=True):
        return preprocessor.fit_transform(X) if fit else preprocessor.transform(X)

    return preprocessor, preprocess

# 4. Model Training Function
def train_and_evaluate(X, y, model, params=None):
    if params:
        grid = GridSearchCV(model, params, cv=5, scoring='roc_auc', n_jobs=-1, error_score='raise')
        grid.fit(X, y)
        best = grid.best_estimator_
    else:
        best = model
        best.fit(X, y)
    y_pred = best.predict(X)
    print(classification_report(y, y_pred))
    print("ROC AUC:", roc_auc_score(y, best.predict_proba(X)[:,1]))
    return best

# 5. Ensemble Prediction via Cross Validation
def cross_val_predict_average(model, X, y, X_test, folds=5):
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)
    test_preds = np.zeros((X_test.shape[0],))
    for train_idx, _ in skf.split(X, y):
        X_tr, y_tr = X[train_idx], y[train_idx]
        model_ = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
        model_.fit(X_tr, y_tr)
        test_preds += model_.predict_proba(X_test)[:,1] / folds
    return test_preds

# 6. Main Execution
if __name__ == '__main__':
    train_df, test_df = load_data('train.csv', 'test.csv')

    # Feature engineering
    for df in [train_df, test_df]:
        df['BMI_GLU'] = df['BMXBMI'] * df['LBXGLU']
        df['INSULIN_RATIO'] = df['LBXIN'] / (df['LBXGLU'] + 1)
        df['GLU_TOL_RATIO'] = df['LBXGLU'] / (df['LBXGLT'] + 1)
        df['INSULIN_GLU_RATIO'] = df['LBXIN'] / (df['LBXGLU'] + 1)
        df['ACTIVITY_DIABETES'] = df['PAQ605'].fillna(0) * df['DIQ010'].fillna(0)

    train_df = clean_data(train_df)
    test_df = clean_data(test_df)  # Avoid filtering rows in test set

    train_df = train_df.dropna(subset=['age_group'])
    train_df['age_group'] = train_df['age_group'].map({'Adult': 0, 'Senior': 1})

    X_train_full = train_df.drop(['SEQN', 'age_group'], axis=1)
    y_train_full = train_df['age_group'].astype(int)
    X_test_feat = test_df.drop(['SEQN'], axis=1)

    preprocessor, preprocess_fn = make_pipeline()
    X_train_trans = preprocess_fn(X_train_full, fit=True)
    X_test_trans = preprocess_fn(X_test_feat, fit=False)

    X_train_trans = np.nan_to_num(X_train_trans)
    X_test_trans = np.nan_to_num(X_test_trans)

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_trans, y_train_full, test_size=0.2, stratify=y_train_full, random_state=42
    )

    xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
    xgb_params = {
        'n_estimators': [100, 200, 300, 500],
        'max_depth': [3, 5, 7, 9],
        'learning_rate': [0.01, 0.05, 0.1],
        'subsample': [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0]
    }

    print("Training XGBoost...")
    best_model = train_and_evaluate(X_tr, y_tr, xgb, xgb_params)

    print("Validation Metrics:")
    y_val_pred = best_model.predict(X_val)
    print(classification_report(y_val, y_val_pred))
    print("ROC AUC:", roc_auc_score(y_val, best_model.predict_proba(X_val)[:,1]))

    print("Generating ensemble prediction on test set...")
    test_probs = cross_val_predict_average(best_model, X_train_trans, y_train_full.to_numpy(), X_test_trans, folds=5)
    test_preds = (test_probs > 0.5).astype(int)

    submission = pd.DataFrame({
        'age_group': test_preds
    })
    submission.to_csv('submission.csv', index=False)
    print("submission.csv created")


Training XGBoost...


/tmp/ipython-input-3-2512837995.py:100: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['age_group'] = train_df['age_group'].map({'Adult': 0, 'Senior': 1})
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [08:29:49] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


              precision    recall  f1-score   support

           0       0.87      0.99      0.93      1310
           1       0.76      0.24      0.37       251

    accuracy                           0.87      1561
   macro avg       0.82      0.61      0.65      1561
weighted avg       0.85      0.87      0.84      1561

ROC AUC: 0.880292570177306
Validation Metrics:
              precision    recall  f1-score   support

           0       0.85      0.97      0.90       328
           1       0.33      0.08      0.13        63

    accuracy                           0.83       391
   macro avg       0.59      0.52      0.52       391
weighted avg       0.76      0.83      0.78       391

ROC AUC: 0.6894115369725126
Generating ensemble prediction on test set...


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [08:29:49] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


submission.csv created


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.ensemble import IsolationForest
from xgboost import XGBClassifier

# 1. Load Data
def load_data(train_path, test_path):
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    return train, test

# 2. Data Cleaning / Outlier Removal (without dropping NaNs)
def clean_data(df):
    num_feats = ['PAQ605','BMXBMI','LBXGLU','LBXGLT','LBXIN']
    df_copy = df.copy()
    df_iso = df_copy[num_feats].copy()
    for col in num_feats:
        df_iso[col] = df_iso[col].fillna(df_iso[col].median())
    mask = IsolationForest(contamination=0.02, random_state=42).fit_predict(df_iso)
    # No filtering: preserve all rows

    for col in num_feats:
        if col in df_copy.columns:
            low, high = df_copy[col].quantile([0.01, 0.99])
            df_copy[col] = df_copy[col].clip(lower=low, upper=high)

    return df_copy

# 3. Preprocessing Pipeline (No PCA)
def make_pipeline():
    num_feats = ['PAQ605','BMXBMI','LBXGLU','LBXGLT','LBXIN','BMI_GLU','INSULIN_RATIO','GLU_TOL_RATIO','INSULIN_GLU_RATIO','ACTIVITY_DIABETES','HIGH_GLUCOSE','HIGH_BMI','LOW_ACTIVITY','INSULIN_DEFICIENT']
    cat_feats = ['RIAGENDR','DIQ010']

    num_pipe = Pipeline([
        ('imputer', KNNImputer(n_neighbors=5)),
        ('scaler', StandardScaler())
    ])
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ])

    from sklearn.compose import ColumnTransformer
    preprocessor = ColumnTransformer([
        ('num', num_pipe, num_feats),
        ('cat', cat_pipe, cat_feats)
    ])

    def preprocess(X, fit=True):
        return preprocessor.fit_transform(X) if fit else preprocessor.transform(X)

    return preprocessor, preprocess

# 4. Model Training Function
def train_and_evaluate(X, y, model, params=None):
    if params:
        grid = GridSearchCV(model, params, cv=5, scoring='roc_auc', n_jobs=-1, error_score='raise')
        grid.fit(X, y)
        best = grid.best_estimator_
    else:
        best = model
        from sklearn.utils.class_weight import compute_sample_weight
        sample_weights = compute_sample_weight(class_weight='balanced', y=y)
        best.fit(X, y, sample_weight=sample_weights)
    y_pred = best.predict(X)
    print(classification_report(y, y_pred))
    print("ROC AUC:", roc_auc_score(y, best.predict_proba(X)[:,1]))
    return best

# 5. Ensemble Prediction via Cross Validation
def cross_val_predict_average(model, X, y, X_test, folds=5):
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)
    test_preds = np.zeros((X_test.shape[0],))
    for train_idx, _ in skf.split(X, y):
        X_tr, y_tr = X[train_idx], y[train_idx]
        model_ = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
        model_.fit(X_tr, y_tr)
        test_preds += model_.predict_proba(X_test)[:,1] / folds
    return test_preds

# 6. Main Execution
if __name__ == '__main__':
    train_df, test_df = load_data('train.csv', 'test.csv')

    # Feature engineering
    for df in [train_df, test_df]:  # Enhanced with additional binary features
        df['BMI_GLU'] = df['BMXBMI'] * df['LBXGLU']
        df['INSULIN_RATIO'] = df['LBXIN'] / (df['LBXGLU'] + 1)
        df['GLU_TOL_RATIO'] = df['LBXGLU'] / (df['LBXGLT'] + 1)
        df['INSULIN_GLU_RATIO'] = df['LBXIN'] / (df['LBXGLU'] + 1)
        df['ACTIVITY_DIABETES'] = df['PAQ605'].fillna(0) * df['DIQ010'].fillna(0)
        df['HIGH_GLUCOSE'] = (df['LBXGLU'] > 140).astype(int)
        df['HIGH_BMI'] = (df['BMXBMI'] > 30).astype(int)
        df['LOW_ACTIVITY'] = (df['PAQ605'] == 2).astype(int)
        df['INSULIN_DEFICIENT'] = (df['LBXIN'] < 5).astype(int)

    train_df = clean_data(train_df)
    test_df = clean_data(test_df)  # Avoid filtering rows in test set

    train_df = train_df.dropna(subset=['age_group'])
    train_df['age_group'] = train_df['age_group'].map({'Adult': 0, 'Senior': 1})

    X_train_full = train_df.drop(['SEQN', 'age_group'], axis=1)
    y_train_full = train_df['age_group'].astype(int)
    X_test_feat = test_df.drop(['SEQN'], axis=1)

    preprocessor, preprocess_fn = make_pipeline()
    X_train_trans = preprocess_fn(X_train_full, fit=True)
    X_test_trans = preprocess_fn(X_test_feat, fit=False)

    X_train_trans = np.nan_to_num(X_train_trans)
    X_test_trans = np.nan_to_num(X_test_trans)

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_trans, y_train_full, test_size=0.2, stratify=y_train_full, random_state=42
    )

    xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
    xgb_params = {
        'n_estimators': [100, 200, 300, 500],
        'max_depth': [3, 5, 7, 9],
        'learning_rate': [0.01, 0.05, 0.1],
        'subsample': [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0]
    }

    print("Training XGBoost...")
    best_model = train_and_evaluate(X_tr, y_tr, xgb, xgb_params)

    print("Validation Metrics:")
    y_val_pred = best_model.predict(X_val)
    print(classification_report(y_val, y_val_pred))
    print("ROC AUC:", roc_auc_score(y_val, best_model.predict_proba(X_val)[:,1]))

    print("Generating ensemble prediction on test set...")
    test_probs = cross_val_predict_average(best_model, X_train_trans, y_train_full.to_numpy(), X_test_trans, folds=5)
    test_preds = (test_probs > 0.5).astype(int)

    submission = pd.DataFrame({
        'age_group': test_preds
    })
    submission.to_csv('submission.csv', index=False)
    print("submission.csv created")


/tmp/ipython-input-4-3356178770.py:106: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['age_group'] = train_df['age_group'].map({'Adult': 0, 'Senior': 1})


Training XGBoost...


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [08:42:37] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


              precision    recall  f1-score   support

           0       0.87      0.98      0.92      1310
           1       0.73      0.24      0.36       251

    accuracy                           0.86      1561
   macro avg       0.80      0.61      0.64      1561
weighted avg       0.85      0.86      0.83      1561

ROC AUC: 0.8775675922265138
Validation Metrics:
              precision    recall  f1-score   support

           0       0.85      0.97      0.90       328
           1       0.35      0.10      0.15        63

    accuracy                           0.83       391
   macro avg       0.60      0.53      0.53       391
weighted avg       0.77      0.83      0.78       391

ROC AUC: 0.6922183507549361
Generating ensemble prediction on test set...


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [08:42:38] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


submission.csv created


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, QuantileTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, confusion_matrix, f1_score
from sklearn.ensemble import IsolationForest, RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from imblearn.over_sampling import BorderlineSMOTE, ADASYN, SMOTE
from imblearn.combine import SMOTEENN, SMOTETomek
from imblearn.under_sampling import EditedNearestNeighbours
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.feature_selection import SelectKBest, f_classif, RFE
import warnings
warnings.filterwarnings('ignore')

# 1. Load Data
def load_data(train_path, test_path):
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    return train, test

# 2. SIGNIFICANTLY Enhanced Feature Engineering
def enhanced_feature_engineering(df):
    """Add comprehensive age-proxy features with medical domain knowledge"""

    # Original interaction features
    df['BMI_GLU'] = df['BMXBMI'] * df['LBXGLU']
    df['INSULIN_RATIO'] = df['LBXIN'] / (df['LBXGLU'] + 1)
    df['GLU_TOL_RATIO'] = df['LBXGLU'] / (df['LBXGLT'] + 1)
    df['INSULIN_GLU_RATIO'] = df['LBXIN'] / (df['LBXGLU'] + 1)
    df['ACTIVITY_DIABETES'] = df['PAQ605'].fillna(0) * df['DIQ010'].fillna(0)

    # Enhanced binary thresholds with clinical knowledge
    df['HIGH_GLUCOSE'] = (df['LBXGLU'] > 140).astype(int)
    df['VERY_HIGH_GLUCOSE'] = (df['LBXGLU'] > 180).astype(int)
    df['PREDIABETIC_GLUCOSE'] = ((df['LBXGLU'] >= 100) & (df['LBXGLU'] < 126)).astype(int)
    df['DIABETIC_GLUCOSE'] = (df['LBXGLU'] >= 126).astype(int)

    df['HIGH_BMI'] = (df['BMXBMI'] > 30).astype(int)
    df['OBESE_CLASS2'] = (df['BMXBMI'] > 35).astype(int)
    df['OBESE_CLASS3'] = (df['BMXBMI'] > 40).astype(int)
    df['UNDERWEIGHT'] = (df['BMXBMI'] < 18.5).astype(int)
    df['OVERWEIGHT'] = ((df['BMXBMI'] >= 25) & (df['BMXBMI'] < 30)).astype(int)

    df['LOW_ACTIVITY'] = (df['PAQ605'] == 2).astype(int)
    df['INSULIN_DEFICIENT'] = (df['LBXIN'] < 5).astype(int)
    df['HIGH_INSULIN'] = (df['LBXIN'] > 20).astype(int)
    df['VERY_HIGH_INSULIN'] = (df['LBXIN'] > 30).astype(int)

    # Advanced metabolic patterns (age-related)
    df['METABOLIC_SYNDROME'] = ((df['BMXBMI'] > 30) &
                               (df['LBXGLU'] > 100) &
                               (df['DIQ010'] == 1)).astype(int)

    df['INSULIN_RESISTANCE'] = ((df['LBXIN'] > 15) & (df['LBXGLU'] > 100)).astype(int)
    df['SEVERE_INSULIN_RESISTANCE'] = ((df['LBXIN'] > 25) & (df['LBXGLU'] > 120)).astype(int)

    # Gender-specific patterns
    df['FEMALE_HIGH_GLUCOSE'] = ((df['RIAGENDR'] == 2) & (df['LBXGLU'] > 120)).astype(int)
    df['MALE_HIGH_BMI'] = ((df['RIAGENDR'] == 1) & (df['BMXBMI'] > 28)).astype(int)
    df['FEMALE_DIABETES'] = ((df['RIAGENDR'] == 2) & (df['DIQ010'] == 1)).astype(int)
    df['MALE_DIABETES'] = ((df['RIAGENDR'] == 1) & (df['DIQ010'] == 1)).astype(int)

    # Complex multi-condition patterns
    df['SEDENTARY_DIABETIC'] = ((df['PAQ605'] == 2) & (df['DIQ010'] == 1)).astype(int)
    df['ACTIVE_DIABETIC'] = ((df['PAQ605'] == 1) & (df['DIQ010'] == 1)).astype(int)
    df['DIABETES_LOW_ACTIVITY'] = ((df['DIQ010'] == 1) & (df['PAQ605'] == 2)).astype(int)

    # Glucose tolerance patterns
    df['POOR_GLUCOSE_TOLERANCE'] = (df['LBXGLT'] > 200).astype(int)
    df['IMPAIRED_GLUCOSE_TOLERANCE'] = ((df['LBXGLT'] >= 140) & (df['LBXGLT'] < 200)).astype(int)
    df['NORMAL_GLUCOSE_TOLERANCE'] = (df['LBXGLT'] < 140).astype(int)

    # Advanced glucose-insulin patterns
    df['HIGH_GLUCOSE_LOW_INSULIN'] = ((df['LBXGLU'] > 126) & (df['LBXIN'] < 10)).astype(int)
    df['NORMAL_GLUCOSE_HIGH_INSULIN'] = ((df['LBXGLU'] < 100) & (df['LBXIN'] > 20)).astype(int)
    df['HIGH_GLUCOSE_HIGH_INSULIN'] = ((df['LBXGLU'] > 126) & (df['LBXIN'] > 20)).astype(int)

    # Ratios and normalized features
    df['BMI_INSULIN_RATIO'] = df['BMXBMI'] / (df['LBXIN'] + 1)
    df['GLUCOSE_TOLERANCE_DIFF'] = df['LBXGLT'] - df['LBXGLU']
    df['INSULIN_GLUCOSE_PRODUCT'] = df['LBXIN'] * df['LBXGLU']
    df['BMI_GLUCOSE_RATIO'] = df['BMXBMI'] / (df['LBXGLU'] + 1)

    # Binned features (age-related distributions)
    df['BMI_CATEGORY'] = pd.cut(df['BMXBMI'], bins=[0, 18.5, 25, 30, 35, 100],
                               labels=[0, 1, 2, 3, 4], include_lowest=True).astype(float)
    df['GLUCOSE_CATEGORY'] = pd.cut(df['LBXGLU'], bins=[0, 100, 126, 180, 1000],
                                   labels=[0, 1, 2, 3], include_lowest=True).astype(float)
    df['INSULIN_CATEGORY'] = pd.cut(df['LBXIN'], bins=[0, 10, 20, 30, 1000],
                                   labels=[0, 1, 2, 3], include_lowest=True).astype(float)

    # Activity-health combinations
    df['INACTIVE_OBESE'] = ((df['PAQ605'] == 2) & (df['BMXBMI'] > 30)).astype(int)
    df['ACTIVE_HEALTHY_WEIGHT'] = ((df['PAQ605'] == 1) & (df['BMXBMI'] < 25)).astype(int)

    # Count-based features
    df['HIGH_RISK_COUNT'] = (df[['HIGH_GLUCOSE', 'HIGH_BMI', 'LOW_ACTIVITY', 'DIQ010']].fillna(0).sum(axis=1))
    df['METABOLIC_RISK_SCORE'] = (df[['DIABETIC_GLUCOSE', 'OBESE_CLASS2', 'HIGH_INSULIN', 'LOW_ACTIVITY']].fillna(0).sum(axis=1))

    return df

# 3. Improved Data Cleaning
def clean_data(df):
    num_feats = ['PAQ605','BMXBMI','LBXGLU','LBXGLT','LBXIN']
    df_copy = df.copy()

    # More conservative outlier removal
    for col in num_feats:
        if col in df_copy.columns:
            Q1 = df_copy[col].quantile(0.05)
            Q3 = df_copy[col].quantile(0.95)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            df_copy[col] = df_copy[col].clip(lower=lower_bound, upper=upper_bound)

    return df_copy

# 4. Enhanced Preprocessing Pipeline
def make_pipeline():
    # Get all numeric features
    num_feats = ['PAQ605','BMXBMI','LBXGLU','LBXGLT','LBXIN','BMI_GLU','INSULIN_RATIO',
                'GLU_TOL_RATIO','INSULIN_GLU_RATIO','ACTIVITY_DIABETES','HIGH_GLUCOSE',
                'VERY_HIGH_GLUCOSE','PREDIABETIC_GLUCOSE','DIABETIC_GLUCOSE',
                'HIGH_BMI','OBESE_CLASS2','OBESE_CLASS3','UNDERWEIGHT','OVERWEIGHT',
                'LOW_ACTIVITY','INSULIN_DEFICIENT','HIGH_INSULIN','VERY_HIGH_INSULIN',
                'METABOLIC_SYNDROME','INSULIN_RESISTANCE','SEVERE_INSULIN_RESISTANCE',
                'FEMALE_HIGH_GLUCOSE','MALE_HIGH_BMI','FEMALE_DIABETES','MALE_DIABETES',
                'SEDENTARY_DIABETIC','ACTIVE_DIABETIC','DIABETES_LOW_ACTIVITY',
                'POOR_GLUCOSE_TOLERANCE','IMPAIRED_GLUCOSE_TOLERANCE','NORMAL_GLUCOSE_TOLERANCE',
                'HIGH_GLUCOSE_LOW_INSULIN','NORMAL_GLUCOSE_HIGH_INSULIN','HIGH_GLUCOSE_HIGH_INSULIN',
                'BMI_INSULIN_RATIO','GLUCOSE_TOLERANCE_DIFF','INSULIN_GLUCOSE_PRODUCT',
                'BMI_GLUCOSE_RATIO','BMI_CATEGORY','GLUCOSE_CATEGORY','INSULIN_CATEGORY',
                'INACTIVE_OBESE','ACTIVE_HEALTHY_WEIGHT','HIGH_RISK_COUNT','METABOLIC_RISK_SCORE']

    cat_feats = ['RIAGENDR','DIQ010']

    # Use QuantileTransformer for better distribution handling
    num_pipe = Pipeline([
        ('imputer', KNNImputer(n_neighbors=7)),
        ('scaler', QuantileTransformer(output_distribution='normal', random_state=42))
    ])

    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ])

    from sklearn.compose import ColumnTransformer
    preprocessor = ColumnTransformer([
        ('num', num_pipe, num_feats),
        ('cat', cat_pipe, cat_feats)
    ])

    return preprocessor

# 5. Enhanced F1-Optimized Threshold Finding
def find_optimal_f1_threshold(model, X_val, y_val, min_precision=0.25):
    """Find threshold that maximizes F1 score with minimum precision constraint"""
    probabilities = model.predict_proba(X_val)[:, 1]

    best_threshold = 0.5
    best_f1 = 0
    best_recall = 0
    best_precision = 0

    # Search for optimal F1 threshold with precision constraint
    for threshold in np.arange(0.1, 0.9, 0.01):
        y_pred_thresh = (probabilities > threshold).astype(int)

        cm = confusion_matrix(y_val, y_pred_thresh)
        if cm.shape == (2, 2) and (cm[1,1] + cm[1,0]) > 0 and (cm[1,1] + cm[0,1]) > 0:
            recall = cm[1,1] / (cm[1,1] + cm[1,0])
            precision = cm[1,1] / (cm[1,1] + cm[0,1])

            # Only consider thresholds that meet minimum precision
            if precision >= min_precision:
                f1 = 2 * (precision * recall) / (precision + recall)

                if f1 > best_f1:
                    best_f1 = f1
                    best_threshold = threshold
                    best_recall = recall
                    best_precision = precision

    print(f"Optimal F1 threshold: {best_threshold:.3f}")
    print(f"Senior Recall: {best_recall:.3f}, Precision: {best_precision:.3f}, F1: {best_f1:.3f}")
    return best_threshold

# 6. F1-Focused Model Training with Aggressive Strategies
def train_enhanced_models(X_train, y_train, X_val, y_val):
    """Train models specifically optimized for F1 score"""

    class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    scale_pos_weight = class_weights[1] / class_weights[0]

    models = {}

    # Strategy 1: More aggressive SMOTE for better recall
    print("Applying Aggressive SMOTE...")
    smote_aggressive = SMOTE(random_state=42, sampling_strategy=0.8, k_neighbors=5)
    X_train_smote_agg, y_train_smote_agg = smote_aggressive.fit_resample(X_train, y_train)
    print(f"After Aggressive SMOTE: Adult={np.sum(y_train_smote_agg==0)}, Senior={np.sum(y_train_smote_agg==1)}")

    # Strategy 2: Borderline SMOTE for hard examples
    print("Applying Borderline SMOTE...")
    borderline_smote = BorderlineSMOTE(random_state=42, sampling_strategy=0.75, k_neighbors=3)
    X_train_borderline, y_train_borderline = borderline_smote.fit_resample(X_train, y_train)
    print(f"After Borderline SMOTE: Adult={np.sum(y_train_borderline==0)}, Senior={np.sum(y_train_borderline==1)}")

    # Strategy 3: Conservative SMOTE for precision
    print("Applying Conservative SMOTE...")
    smote_conservative = SMOTE(random_state=42, sampling_strategy=0.6)
    X_train_smote_cons, y_train_smote_cons = smote_conservative.fit_resample(X_train, y_train)
    print(f"After Conservative SMOTE: Adult={np.sum(y_train_smote_cons==0)}, Senior={np.sum(y_train_smote_cons==1)}")

    # Model 1: F1-Optimized LightGBM (High Recall)
    print("Training F1-Optimized LightGBM (High Recall)...")
    lgb_recall = LGBMClassifier(
        objective='binary',
        boosting_type='gbdt',
        num_leaves=20,
        learning_rate=0.08,
        feature_fraction=0.9,
        bagging_fraction=0.9,
        bagging_freq=3,
        min_child_samples=10,
        lambda_l1=0.01,
        lambda_l2=0.1,
        scale_pos_weight=scale_pos_weight * 1.5,  # More aggressive
        random_state=42,
        verbosity=-1,
        n_estimators=250
    )
    lgb_recall.fit(X_train_smote_agg, y_train_smote_agg)
    models['lgb_recall'] = lgb_recall

    # Model 2: F1-Optimized LightGBM (Balanced)
    print("Training F1-Optimized LightGBM (Balanced)...")
    lgb_balanced = LGBMClassifier(
        objective='binary',
        boosting_type='gbdt',
        num_leaves=25,
        learning_rate=0.06,
        feature_fraction=0.8,
        bagging_fraction=0.8,
        bagging_freq=5,
        min_child_samples=15,
        lambda_l1=0.05,
        lambda_l2=0.2,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        verbosity=-1,
        n_estimators=300
    )
    lgb_balanced.fit(X_train_borderline, y_train_borderline)
    models['lgb_balanced'] = lgb_balanced

    # Model 3: F1-Optimized XGBoost (High Recall)
    print("Training F1-Optimized XGBoost (High Recall)...")
    xgb_recall = XGBClassifier(
        scale_pos_weight=scale_pos_weight * 1.8,  # Very aggressive
        max_depth=5,
        learning_rate=0.1,
        n_estimators=250,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_alpha=0.01,
        reg_lambda=0.3,
        gamma=0.05,
        min_child_weight=1,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    )
    xgb_recall.fit(X_train_smote_agg, y_train_smote_agg)
    models['xgb_recall'] = xgb_recall

    # Model 4: F1-Optimized XGBoost (Precision Focus)
    print("Training F1-Optimized XGBoost (Precision Focus)...")
    xgb_precision = XGBClassifier(
        scale_pos_weight=scale_pos_weight * 0.8,  # Less aggressive
        max_depth=7,
        learning_rate=0.05,
        n_estimators=400,
        subsample=0.7,
        colsample_bytree=0.7,
        reg_alpha=0.2,
        reg_lambda=1.0,
        gamma=0.2,
        min_child_weight=5,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    )
    xgb_precision.fit(X_train_smote_cons, y_train_smote_cons)
    models['xgb_precision'] = xgb_precision

    # Model 5: F1-Optimized Random Forest
    print("Training F1-Optimized Random Forest...")
    rf_f1 = RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_split=2,
        min_samples_leaf=1,
        class_weight={0: 1, 1: scale_pos_weight * 1.3},  # Custom weights
        max_features='log2',
        random_state=42,
        n_jobs=-1
    )
    rf_f1.fit(X_train_borderline, y_train_borderline)
    models['rf_f1'] = rf_f1

    # Model 6: Calibrated Logistic Regression
    print("Training Calibrated Logistic Regression...")
    from sklearn.calibration import CalibratedClassifierCV

    lr_base = LogisticRegression(
        class_weight={0: 1, 1: scale_pos_weight * 1.2},
        C=1.0,
        penalty='l2',
        solver='lbfgs',
        max_iter=1000,
        random_state=42
    )
    lr_calibrated = CalibratedClassifierCV(lr_base, method='isotonic', cv=3)
    lr_calibrated.fit(X_train_smote_agg, y_train_smote_agg)
    models['lr_calibrated'] = lr_calibrated

    # Evaluate each model
    for name, model in models.items():
        print(f"\n{name.upper()} Validation Results:")
        y_pred = model.predict(X_val)
        y_proba = model.predict_proba(X_val)[:, 1]
        print(classification_report(y_val, y_pred))
        print(f"ROC AUC: {roc_auc_score(y_val, y_proba):.4f}")

        cm = confusion_matrix(y_val, y_pred)
        if cm.shape == (2, 2) and (cm[1,1] + cm[1,0]) > 0 and (cm[1,1] + cm[0,1]) > 0:
            senior_recall = cm[1,1] / (cm[1,1] + cm[1,0])
            senior_precision = cm[1,1] / (cm[1,1] + cm[0,1])
            senior_f1 = 2 * (senior_precision * senior_recall) / (senior_precision + senior_recall)
            print(f"Senior Metrics - Recall: {senior_recall:.4f}, Precision: {senior_precision:.4f}, F1: {senior_f1:.4f}")

    return models

# 7. F1-Focused Ensemble with Dynamic Weighting
def ensemble_predict(models, X_test, X_val, y_val):
    """Create F1-optimized ensemble with dynamic model weighting"""

    # Step 1: Evaluate each model's F1 performance with different thresholds
    model_performance = {}
    model_probabilities = {}

    for name, model in models.items():
        val_proba = model.predict_proba(X_val)[:, 1]
        test_proba = model.predict_proba(X_test)[:, 1]

        # Find best F1 threshold for this model (more lenient precision requirement)
        best_f1 = 0
        best_threshold = 0.5
        best_recall = 0
        best_precision = 0

        for threshold in np.arange(0.15, 0.85, 0.02):  # Wider range
            val_pred = (val_proba > threshold).astype(int)
            cm = confusion_matrix(y_val, val_pred)
            if cm.shape == (2, 2) and (cm[1,1] + cm[1,0]) > 0 and (cm[1,1] + cm[0,1]) > 0:
                recall = cm[1,1] / (cm[1,1] + cm[1,0])
                precision = cm[1,1] / (cm[1,1] + cm[0,1])

                # More lenient precision requirement for better F1
                if precision >= 0.20 and recall >= 0.30:  # Minimum thresholds
                    f1 = 2 * (precision * recall) / (precision + recall)
                    if f1 > best_f1:
                        best_f1 = f1
                        best_threshold = threshold
                        best_recall = recall
                        best_precision = precision

        model_performance[name] = {
            'f1': best_f1,
            'threshold': best_threshold,
            'recall': best_recall,
            'precision': best_precision
        }
        model_probabilities[name] = test_proba

        print(f"{name} - F1: {best_f1:.4f}, Threshold: {best_threshold:.3f}, "
              f"Recall: {best_recall:.3f}, Precision: {best_precision:.3f}")

    # Step 2: Create dynamic weights based on F1 performance
    f1_scores = [perf['f1'] for perf in model_performance.values()]

    # Use exponential weighting to favor better models
    weights = np.exp(np.array(f1_scores) * 4)  # Higher temperature for more aggressive weighting
    weights = weights / np.sum(weights)

    print(f"\nDynamic model weights:")
    for i, (name, weight) in enumerate(zip(model_performance.keys(), weights)):
        print(f"  {name}: {weight:.3f}")

    # Step 3: Create weighted ensemble probabilities
    ensemble_proba = np.zeros(len(X_test))
    for i, (name, proba) in enumerate(model_probabilities.items()):
        ensemble_proba += weights[i] * proba

    # Step 4: Find optimal threshold for the ensemble
    ensemble_val_proba = np.zeros(len(X_val))
    for i, (name, model) in enumerate(models.items()):
        val_proba = model.predict_proba(X_val)[:, 1]
        ensemble_val_proba += weights[i] * val_proba

    # Optimize threshold for F1 with more aggressive parameters
    best_f1 = 0
    best_threshold = 0.5
    best_recall = 0
    best_precision = 0

    for threshold in np.arange(0.1, 0.8, 0.01):
        val_pred = (ensemble_val_proba > threshold).astype(int)
        cm = confusion_matrix(y_val, val_pred)

        if cm.shape == (2, 2) and (cm[1,1] + cm[1,0]) > 0 and (cm[1,1] + cm[0,1]) > 0:
            recall = cm[1,1] / (cm[1,1] + cm[1,0])
            precision = cm[1,1] / (cm[1,1] + cm[0,1])

            # More aggressive F1 optimization
            if precision >= 0.22 and recall >= 0.35:  # Balanced requirements
                f1 = 2 * (precision * recall) / (precision + recall)
                if f1 > best_f1:
                    best_f1 = f1
                    best_threshold = threshold
                    best_recall = recall
                    best_precision = precision

    print(f"\nEnsemble Performance:")
    print(f"Optimal threshold: {best_threshold:.3f}")
    print(f"F1: {best_f1:.4f}, Recall: {best_recall:.3f}, Precision: {best_precision:.3f}")

    # Apply threshold
    ensemble_pred = (ensemble_proba > best_threshold).astype(int)

    return ensemble_pred, ensemble_proba

def find_optimal_f1_threshold_direct(probabilities, y_true, min_precision=0.20):
    """Direct threshold optimization with more aggressive F1 focus"""
    best_threshold = 0.5
    best_f1 = 0

    for threshold in np.arange(0.1, 0.9, 0.01):
        y_pred_thresh = (probabilities > threshold).astype(int)
        cm = confusion_matrix(y_true, y_pred_thresh)

        if cm.shape == (2, 2) and (cm[1,1] + cm[1,0]) > 0 and (cm[1,1] + cm[0,1]) > 0:
            recall = cm[1,1] / (cm[1,1] + cm[1,0])
            precision = cm[1,1] / (cm[1,1] + cm[0,1])

            # Prioritize F1 over precision constraint
            if precision >= min_precision and recall >= 0.30:
                f1 = 2 * (precision * recall) / (precision + recall)
                if f1 > best_f1:
                    best_f1 = f1
                    best_threshold = threshold

    print(f"Final ensemble threshold: {best_threshold:.3f}, F1: {best_f1:.3f}")
    return best_threshold

# 8. Main Execution
if __name__ == '__main__':
    print("Loading data...")
    train_df, test_df = load_data('train.csv', 'test.csv')

    print("Enhanced feature engineering...")
    train_df = enhanced_feature_engineering(train_df)
    test_df = enhanced_feature_engineering(test_df)

    print("Cleaning data...")
    train_df = clean_data(train_df)
    test_df = clean_data(test_df)

    # Drop missing target labels only
    train_df = train_df.dropna(subset=['age_group'])

    # Map age_group to 0/1
    train_df['age_group'] = train_df['age_group'].map({'Adult': 0, 'Senior': 1})

    print(f"Class distribution: Adult={np.sum(train_df['age_group']==0)}, Senior={np.sum(train_df['age_group']==1)}")

    X_train_full = train_df.drop(['SEQN', 'age_group'], axis=1)
    y_train_full = train_df['age_group'].astype(int)
    X_test_feat = test_df.drop(['SEQN'], axis=1)

    print("Preprocessing...")
    preprocessor = make_pipeline()

    X_train_trans = preprocessor.fit_transform(X_train_full)
    X_test_trans = preprocessor.transform(X_test_feat)

    # Final NaN check and replace
    X_train_trans = np.nan_to_num(X_train_trans)
    X_test_trans = np.nan_to_num(X_test_trans)

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_trans, y_train_full, test_size=0.2, stratify=y_train_full, random_state=42
    )

    print("Training enhanced models...")
    models = train_enhanced_models(X_tr, y_tr, X_val, y_val)

    print("Generating ensemble predictions...")
    test_predictions, test_probabilities = ensemble_predict(models, X_test_trans, X_val, y_val)

    # Final validation
    print(f"\nFinal prediction distribution: Adult={np.sum(test_predictions==0)}, Senior={np.sum(test_predictions==1)}")
    print(f"Senior percentage: {np.mean(test_predictions)*100:.1f}%")

    submission = pd.DataFrame({
        'age_group': test_predictions
    })
    submission.to_csv('submission.csv', index=False)
    print("Enhanced submission.csv created!")

Loading data...
Enhanced feature engineering...
Cleaning data...
Class distribution: Adult=1638, Senior=314
Preprocessing...
Training enhanced models...
Applying Aggressive SMOTE...
After Aggressive SMOTE: Adult=1310, Senior=1048
Applying Borderline SMOTE...
After Borderline SMOTE: Adult=1310, Senior=982
Applying Conservative SMOTE...
After Conservative SMOTE: Adult=1310, Senior=786
Training F1-Optimized LightGBM (High Recall)...
Training F1-Optimized LightGBM (Balanced)...
Training F1-Optimized XGBoost (High Recall)...
Training F1-Optimized XGBoost (Precision Focus)...
Training F1-Optimized Random Forest...
Training Calibrated Logistic Regression...

LGB_RECALL Validation Results:
              precision    recall  f1-score   support

           0       0.88      0.77      0.82       328
           1       0.26      0.43      0.33        63

    accuracy                           0.72       391
   macro avg       0.57      0.60      0.57       391
weighted avg       0.78      0.72    

In [ ]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import os

print("--- NHANES Age Group Prediction Script ---")

# --- Step 1: Load Raw Data ---
# This script assumes the training and test CSVs are in the same directory.
try:
    print("\n--- Step 1: Loading Raw Data ---")
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')
    print("Train_Data(2).csv and Test_Data(2).csv loaded successfully.")
except FileNotFoundError:
    print("\nFATAL ERROR: Make sure 'Train_Data(2).csv' and 'Test_Data(2).csv' are in the same directory as this script.")
    exit()

# Store SEQN for final submission and define feature/target sets
test_seqn = test_df['SEQN']
X_train_raw = train_df.drop(['SEQN', 'age_group'], axis=1)
y_train = train_df['age_group']
X_test_raw = test_df.drop('SEQN', axis=1)


# --- Step 2: Data Cleaning (Imputation) ---
print("\n--- Step 2: Cleaning Data (Imputing NaNs) ---")
# Use the median for imputation as it's robust to outliers
imputer = SimpleImputer(strategy='median')
# Fit the imputer on the training data only to avoid data leakage
imputer.fit(X_train_raw)
# Transform both datasets
X_train_imputed = imputer.transform(X_train_raw)
X_test_imputed = imputer.transform(X_test_raw)

# Convert back to DataFrame to keep column names
X_train_imputed_df = pd.DataFrame(X_train_imputed, columns=X_train_raw.columns)
X_test_imputed_df = pd.DataFrame(X_test_imputed, columns=X_test_raw.columns)
print("Missing values have been imputed using the median strategy.")


# --- Step 3: Feature Engineering ---
print("\n--- Step 3: Engineering New Features ---")
def create_features(df):
    """Adds new, potentially useful features to the dataframe."""
    # Add a small constant to prevent division by zero
    epsilon = 1e-6
    # Feature 1: Glucose to Insulin Ratio
    df['Glucose_Insulin_Ratio'] = df['LBXGLU'] / (df['LBXIN'] + epsilon)
    # Feature 2: BMI and Glucose interaction term
    df['BMI_Glucose_Interaction'] = df['BMXBMI'] * df['LBXGLU']
    # Feature 3: Indicator for high Glucose Tolerance Test (GTT) result
    df['High_GTT'] = (df['LBXGLT'] > 140).astype(int)
    return df

X_train_featured = create_features(X_train_imputed_df)
X_test_featured = create_features(X_test_imputed_df)
print("New features have been created: 'Glucose_Insulin_Ratio', 'BMI_Glucose_Interaction', 'High_GTT'")


# --- Step 4: Model Training ---
print("\n--- Step 4: Training the Classification Model ---")
# RandomForest is a powerful model.
# class_weight='balanced' is crucial to handle the imbalanced dataset and meet the project's goal.
# random_state=42 ensures the model is reproducible.
model = RandomForestClassifier(class_weight='balanced', random_state=42)
# Train the model on the fully prepared training data
model.fit(X_train_featured, y_train)
print("Model training is complete.")


# --- Step 5: Prediction ---
print("\n--- Step 5: Making Predictions ---")
predictions = model.predict(X_test_featured)
print("Predictions have been generated for the test data.")


# --- Step 6: Submission File Generation ---
print("\n--- Step 6: Generating Final Submission File ---")
submission_df = pd.DataFrame({'SEQN': test_seqn, 'age_group': predictions})
submission_df.to_csv('submission.csv', index=False)

print("\n\nSUCCESS! The 'submission.csv' file has been created in this directory.")
print("\nHere is a preview of the submission file:")
print(submission_df.head())

--- NHANES Age Group Prediction Script ---

--- Step 1: Loading Raw Data ---
Train_Data(2).csv and Test_Data(2).csv loaded successfully.

--- Step 2: Cleaning Data (Imputing NaNs) ---
Missing values have been imputed using the median strategy.

--- Step 3: Engineering New Features ---
New features have been created: 'Glucose_Insulin_Ratio', 'BMI_Glucose_Interaction', 'High_GTT'

--- Step 4: Training the Classification Model ---


ValueError: Input contains NaN